In [4]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchvision


def sigmoid(x):
    return 1/(1 + np.exp(-x))

def sigmoidPrime(a):
    # 'a' is an already-computed sigmoid output, so the derivative is a*(1-a)
    return a * (1 - a)

def forward(input, weight, bias, activate=True):
    z = np.dot(input, weight) + bias
    return sigmoid(z) if activate else z

def softmax(Output):
    e = np.exp(Output - np.max(Output))   # subtract max for numerical stability
    return e / np.sum(e)

def crossEntropyLoss(y, p):
    return -np.sum(y * np.log(p + 1e-12))

def backward(inputVector, y, HiddenOutPut1, HiddenOutPut2, Probs,
             Hidden1Weights, Hidden1Bias,
             Hidden2Weights, Hidden2Bias,
             OutputWeights, OutputBias, lr):
    # softmax + cross-entropy => gradient at the raw output is simply (p - y)
    dOut = Probs - y
    dHidden2 = (OutputWeights.T @ dOut) * sigmoidPrime(HiddenOutPut2)
    dHidden1 = (Hidden2Weights.T @ dHidden2) * sigmoidPrime(HiddenOutPut1)

    # gradient of a layer's weights = outer(delta, that layer's input)
    OutputWeights  -= lr * np.outer(dOut, HiddenOutPut2)
    OutputBias     -= lr * dOut
    Hidden2Weights -= lr * np.outer(dHidden2, HiddenOutPut1)
    Hidden2Bias    -= lr * dHidden2
    Hidden1Weights -= lr * np.outer(dHidden1, inputVector)
    Hidden1Bias    -= lr * dHidden1


hiddensize = 16

train_set = torchvision.datasets.MNIST(root="./data", train=True, download=True)
test_set = torchvision.datasets.MNIST(root="./data", train=False, download=True)

X_train = train_set.data.numpy().reshape(-1, 784) / 255.0
Y_train = train_set.targets.numpy()
X_test = test_set.data.numpy().reshape(-1, 784) / 255.0
Y_test = test_set.targets.numpy()


Hidden1Weights = np.random.randn(hiddensize, 784) * np.sqrt(1/784)
Hidden1Bias = np.zeros(hiddensize)
Hidden2Weights = np.random.randn(hiddensize, hiddensize) * np.sqrt(1/hiddensize)
Hidden2Bias = np.zeros(hiddensize)
OutputWeights = np.random.randn(10, hiddensize) * np.sqrt(1/hiddensize)
OutputBias = np.zeros(10)

epochs = 5
lr = 0.1

for epoch in range(epochs):
    runningLoss = 0.0

    for i in range(len(Y_train)):

        inputVector = X_train[i]
        y = np.zeros(10)
        y[Y_train[i]] = 1

        HiddenOutPut1 = np.zeros(hiddensize)
        HiddenOutPut2 = np.zeros(hiddensize)
        RawOutput = np.zeros(10)

        for j in range(0, hiddensize):
            HiddenOutPut1[j] = forward(inputVector, Hidden1Weights[j, :], Hidden1Bias[j])
        for x in range(0, hiddensize):
            HiddenOutPut2[x] = forward(HiddenOutPut1, Hidden2Weights[x, :], Hidden2Bias[x])
        for k in range(0, 10):
            RawOutput[k] = forward(HiddenOutPut2, OutputWeights[k, :], OutputBias[k], activate=False)

        Probs = softmax(RawOutput)
        Loss = crossEntropyLoss(y, Probs)
        runningLoss += Loss

        backward(inputVector, y, HiddenOutPut1, HiddenOutPut2, Probs,
                 Hidden1Weights, Hidden1Bias,
                 Hidden2Weights, Hidden2Bias,
                 OutputWeights, OutputBias, lr)

    print(f"epoch {epoch + 1}: mean loss = {runningLoss / len(Y_train):.4f}")
    correct = 0
    for i in range(len(Y_test)):
        inputVector = X_test[i]
        for j in range(0, hiddensize):
            HiddenOutPut1[j] = forward(inputVector, Hidden1Weights[j, :], Hidden1Bias[j])
        for x in range(0, hiddensize):
            HiddenOutPut2[x] = forward(HiddenOutPut1, Hidden2Weights[x, :], Hidden2Bias[x])
        for k in range(0, 10):
            RawOutput[k] = forward(HiddenOutPut2, OutputWeights[k, :], OutputBias[k], activate=False)
        correct += (np.argmax(RawOutput) == Y_test[i])
    print(f"  test accuracy = {correct / len(Y_test):.4f}")

epoch 1: mean loss = 0.4061
  test accuracy = 0.8982
epoch 2: mean loss = 0.2789
  test accuracy = 0.9008
epoch 3: mean loss = 0.2509
  test accuracy = 0.8996
epoch 4: mean loss = 0.2414
  test accuracy = 0.9228
epoch 5: mean loss = 0.2263
  test accuracy = 0.9154
